In [1]:
from pathlib import Path
from typing import Dict, Tuple, Any, Callable
import os
if '__file__' not in locals():
    __file__ = os.path.abspath('')
os.chdir(Path(__file__).parent)  # Go to repo root
print("Changed working directory to repo root:", os.getcwd())
import time
import json
import random
from matplotlib import pyplot as plt
import nuscenes
import nuscenes.eval
from nuscenes.eval.common.config import config_factory
import nuscenes.eval.detection
import nuscenes.eval.detection.constants
from nuscenes.eval.detection.data_classes import DetectionMetrics, DetectionBox, DetectionConfig, DetectionMetricDataList
from nuscenes.eval.common.data_classes import EvalBoxes
from nuscenes.eval.common.loaders import load_prediction, load_gt, add_center_dist, filter_eval_boxes
from nuscenes.eval.detection.algo import calc_ap, calc_tp
from nuscenes.eval.detection.constants import TP_METRICS, DETECTION_NAMES, PRETTY_DETECTION_NAMES
from nuscenes.eval.detection.render import class_pr_curve, class_tp_curve, dist_pr_curve, visualize_sample
from nuscenes.eval.common.render import setup_axis
from nuscenes.eval.common.data_classes import EvalBoxes
from nuscenes.eval.common.utils import center_distance, scale_iou, yaw_diff, velocity_l2, attr_acc, cummean
from nuscenes.eval.detection.data_classes import DetectionMetricData
DETECTION_NAMES_DEFAULT = nuscenes.eval.detection.constants.DETECTION_NAMES.copy()
import numpy as np
import yaml
np.Inf = np.inf # Hack for backward compatibility

from src.data.nuscenes import _NuscenesSingleton
from src.labeling.mapping import get_class_mapping, get_mapping_classes

Changed working directory to repo root: /home/esteban/EDGAR/selfsupervised_openworld_od


/home/esteban/.local/lib/python3.10/site-packages/sklearn/utils/__init__.py:16: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 1.22.0)
  from scipy.sparse import issparse


In [2]:
submission_path = "outs/vlm/baseline-mini-100/#out_labels/vlm_baseline-mini_train_8class.json"
with open(submission_path, 'r') as f:
        submission = json.load(f)
mapping_type = submission['mapping_name']
split = submission['split']
data_config = {"root_path": "/home/esteban/data/nuscenes_mini",
  "dataset": "nuscenes",
  "dataset_version": "v1.0-mini"}
nusc: nuscenes.NuScenes = _NuscenesSingleton.get_nuscenes_instance(data_config['dataset_version'], data_config['root_path'])
if data_config['dataset_version'] == 'v1.0-mini':
    split = f"mini_{split}"
config_path = Path("configs/vlm/baseline-mini-100.yaml")
with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
config['exp_name'] = config_path.stem

In [3]:
def _prepare_config(mapping_type) -> DetectionConfig:
    nuscenes.eval.detection.constants.DETECTION_NAMES.clear()
    nuscenes.eval.detection.constants.DETECTION_NAMES.extend(DETECTION_NAMES_DEFAULT) # Otherwise, the config_factory fails
    config: DetectionConfig = config_factory('detection_cvpr_2019')
    config.class_range['vehicle'] = config.class_range['car'] # Vehicle is an unknown class in nuScenes
    mapping_classes = get_mapping_classes()[mapping_type]
    config.class_range = {k: v for k, v in config.class_range.items() if k in mapping_classes}
    config.class_names = config.class_range.keys()
    return config


eval_config = _prepare_config(mapping_type)
test_config = _prepare_config(mapping_type)
for category in test_config.class_range:
    test_config.class_range[category] = 100

print(test_config.class_range)
print(eval_config.class_range)

{'car': 100, 'truck': 100, 'bus': 100, 'trailer': 100, 'construction_vehicle': 100, 'pedestrian': 100, 'motorcycle': 100, 'bicycle': 100}
{'car': 50, 'truck': 50, 'bus': 50, 'trailer': 50, 'construction_vehicle': 50, 'pedestrian': 40, 'motorcycle': 40, 'bicycle': 40}


In [4]:
test_config.class_range

{'car': 100,
 'truck': 100,
 'bus': 100,
 'trailer': 100,
 'construction_vehicle': 100,
 'pedestrian': 100,
 'motorcycle': 100,
 'bicycle': 100}

In [5]:
from src.evaluate.evaluate_nusc import _DetectionEval
EXP_PSEUDO_OUT_PATH: Path = Path('./outs')
def get_out_dir(config: dict, modulename: str) -> Path:
    out_dir = EXP_PSEUDO_OUT_PATH / config['type'] / config['exp_name'] / modulename
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

nusc_eval: _DetectionEval = _DetectionEval(nusc, config=test_config, result_path=submission_path, 
                                               eval_set=split,output_dir=get_out_dir(config, '#out_eval'), mapping_type = mapping_type, 
                                               timestamp_seconds=True)
get_out_dir(config, '#out_eval')
nusc_eval.main(plot_examples=25, render_curves=True)

{'label_aps': defaultdict(<function nuscenes.eval.detection.data_classes.DetectionMetrics.__init__.<locals>.<lambda>()>,
             {'car': defaultdict(float,
                          {0.5: 0.07624754358040141,
                           1.0: 0.13827344228212488,
                           2.0: 0.18543570516987012,
                           4.0: 0.2016022923256442}),
              'truck': defaultdict(float,
                          {0.5: 0.00028130378277385675,
                           1.0: 0.00806973573437221,
                           2.0: 0.02365515373120005,
                           4.0: 0.037462645118550006}),
              'bus': defaultdict(float,
                          {0.5: 0.006368440674956968,
                           1.0: 0.020794575268995474,
                           2.0: 0.06192867097017548,
                           4.0: 0.08027490975599144}),
              'trailer': defaultdict(float,
                          {0.5: 0.0, 1.0: 0.0, 2.0: 0.0, 4.0: 0.0}